# Portfolio Optimization

This notebook uses **Mean-Variance Optimization (Markowitz)** to find optimal portfolio weights that either:
- **Maximise the Sharpe ratio** (best risk-adjusted return)
- **Minimise volatility** (lowest possible risk)

It also generates the **Efficient Frontier** — the set of portfolios that offer the highest expected return for a given level of risk — visualised in 2D and 3D using Monte Carlo random portfolio simulation.

In [ ]:
%pip install yfinance numpy scipy matplotlib plotly pandas -q

In [ ]:
import yfinance as yf
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from scipy.optimize import minimize
from datetime import datetime, timedelta

In [ ]:
# Create your portfolio here
tickers = ['SPY', 'QQQ']
weights = [0.60, 0.40]
portfolio = dict(zip(tickers, weights))
print('Portfolio:', portfolio)

# Date range
start_date = '2000-01-01'
end_date = (datetime.today() - timedelta(days=1)).strftime('%Y-%m-%d')
print('Start Date:', start_date)
print('End Date  :', end_date)

# Risk-free rate
risk_free_rate = 0.02
print('Risk-free Rate:', risk_free_rate)

## Data Loading

In [ ]:
data = yf.download(tickers, start=start_date, end=end_date, auto_adjust=True, progress=False)
if isinstance(data.columns, pd.MultiIndex):
    data = data['Close']

available = [t for t in tickers if t in data.columns]
data = data[available]

returns = data.pct_change(fill_method=None).dropna(how='all').fillna(0)

# Align weights
w_aligned = np.array([weights[tickers.index(t)] for t in available])
w_aligned = w_aligned / w_aligned.sum()

print(f'Available tickers: {available}')
print(f'Aligned weights:   {w_aligned}')
print(f'Returns shape:     {returns.shape}')

## Portfolio Statistics Functions

In [ ]:
n = len(available)

def port_stats(w):
    """Returns (annual_return, annual_volatility, sharpe) for weight vector w."""
    r   = float((returns @ w).mean() * 252)
    vol = float(np.sqrt(w @ (returns.cov() * 252).values @ w))
    sh  = (r - risk_free_rate) / vol if vol > 0 else 0.0
    return r, vol, sh

def neg_sharpe(w):
    r, vol, _ = port_stats(w)
    return -(r - risk_free_rate) / vol if vol > 0 else 0.0

def port_vol(w):
    return port_stats(w)[1]

# Current portfolio stats
r_cur, v_cur, s_cur = port_stats(w_aligned)
print(f'Current portfolio:')
print(f'  Annual Return:     {r_cur:.2%}')
print(f'  Annual Volatility: {v_cur:.2%}')
print(f'  Sharpe Ratio:      {s_cur:.3f}')

## Optimization

Using `scipy.optimize.minimize` with SLSQP (Sequential Least Squares Programming).

In [ ]:
bounds = [(0.0, 1.0)] * n
constraints = {'type': 'eq', 'fun': lambda w: np.sum(w) - 1}
w0 = np.ones(n) / n  # equal weight starting point

# Maximise Sharpe
res_sh = minimize(neg_sharpe, w0, method='SLSQP', bounds=bounds, constraints=constraints)
w_sh = res_sh.x
r_sh, v_sh, s_sh = port_stats(w_sh)

# Minimise Volatility
res_mv = minimize(port_vol, w0, method='SLSQP', bounds=bounds, constraints=constraints)
w_mv = res_mv.x
r_mv, v_mv, s_mv = port_stats(w_mv)

print('=== Max Sharpe Portfolio ===')
print(f'  Annual Return:     {r_sh:.2%}')
print(f'  Annual Volatility: {v_sh:.2%}')
print(f'  Sharpe Ratio:      {s_sh:.3f}')
print('  Weights:')
for t, w in zip(available, w_sh):
    cur = w_aligned[available.index(t)]
    print(f'    {t}: {w:.1%}  (current: {cur:.1%}, change: {w-cur:+.1%})')

print()
print('=== Min Volatility Portfolio ===')
print(f'  Annual Return:     {r_mv:.2%}')
print(f'  Annual Volatility: {v_mv:.2%}')
print(f'  Sharpe Ratio:      {s_mv:.3f}')
print('  Weights:')
for t, w in zip(available, w_mv):
    cur = w_aligned[available.index(t)]
    print(f'    {t}: {w:.1%}  (current: {cur:.1%}, change: {w-cur:+.1%})')

## Efficient Frontier via Monte Carlo Simulation

Generating X amount of random portfolios to map the efficient frontier.

In [ ]:
n_sim = 5000
sim_rets  = np.zeros(n_sim)
sim_vols  = np.zeros(n_sim)
sim_shrps = np.zeros(n_sim)
sim_wmat  = np.zeros((n_sim, n))

rng = np.random.default_rng(42)
for i in range(n_sim):
    w = rng.dirichlet(np.ones(n))
    r, v, s = port_stats(w)
    sim_rets[i]  = r
    sim_vols[i]  = v
    sim_shrps[i] = s
    sim_wmat[i]  = w

print(f'Simulated {n_sim} portfolios')
print(f'Sharpe range: {sim_shrps.min():.2f} – {sim_shrps.max():.2f}')

## Efficient Frontier 2D

In [ ]:
fig, ax = plt.subplots(figsize=(10, 7))

sc = ax.scatter(sim_vols * 100, sim_rets * 100, c=sim_shrps,
                cmap='RdYlGn', alpha=0.5, s=6, linewidths=0)
plt.colorbar(sc, ax=ax, label='Sharpe Ratio')

# Current portfolio
ax.scatter(v_cur * 100, r_cur * 100, marker='o', s=120, color='red',
           zorder=5, label=f'Current (Sharpe={s_cur:.2f})')

# Max Sharpe
ax.scatter(v_sh * 100, r_sh * 100, marker='D', s=80, color='limegreen',
           zorder=5, label=f'Max Sharpe (Sharpe={s_sh:.2f})')

# Min Volatility
ax.scatter(v_mv * 100, r_mv * 100, marker='s', s=80, color='royalblue',
           zorder=5, label='Min Volatility')

ax.set_xlabel('Annual Volatility (%)')
ax.set_ylabel('Annual Return (%)')
ax.set_title(f'Efficient Frontier ({n_sim} Random Portfolios)')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:.0f}%'))
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:.0f}%'))
plt.tight_layout()
plt.show()

## Efficient Frontier 3D (Interactive)

Interactive 3D chart using Plotly. X = volatility, Y = return, Z = Sharpe ratio. Drag to rotate, scroll to zoom.

In [ ]:
import plotly.graph_objects as go

fig3d = go.Figure()

# Scatter cloud
fig3d.add_trace(go.Scatter3d(
    x=sim_vols * 100, y=sim_rets * 100, z=sim_shrps,
    mode='markers',
    marker=dict(size=2, color=sim_shrps, colorscale='RdYlGn', opacity=0.5,
                colorbar=dict(title='Sharpe', thickness=12, len=0.6)),
    name='Simulated Portfolios',
    hovertemplate='Vol: %{x:.1f}%<br>Return: %{y:.1f}%<br>Sharpe: %{z:.2f}<extra></extra>',
))

# Current portfolio
fig3d.add_trace(go.Scatter3d(
    x=[v_cur * 100], y=[r_cur * 100], z=[s_cur],
    mode='markers',
    marker=dict(size=8, color='red', symbol='circle'),
    name=f'Current (Sharpe={s_cur:.2f})',
    hovertemplate=f'Current<br>Vol: {v_cur*100:.1f}%<br>Return: {r_cur*100:.1f}%<br>Sharpe: {s_cur:.2f}<extra></extra>',
))

# Max Sharpe
fig3d.add_trace(go.Scatter3d(
    x=[v_sh * 100], y=[r_sh * 100], z=[s_sh],
    mode='markers',
    marker=dict(size=8, color='limegreen', symbol='diamond'),
    name=f'Max Sharpe ({s_sh:.2f})',
    hovertemplate=f'Max Sharpe<br>Vol: {v_sh*100:.1f}%<br>Return: {r_sh*100:.1f}%<br>Sharpe: {s_sh:.2f}<extra></extra>',
))

# Min Volatility
fig3d.add_trace(go.Scatter3d(
    x=[v_mv * 100], y=[r_mv * 100], z=[s_mv],
    mode='markers',
    marker=dict(size=8, color='royalblue', symbol='square'),
    name='Min Volatility',
    hovertemplate=f'Min Vol<br>Vol: {v_mv*100:.1f}%<br>Return: {r_mv*100:.1f}%<br>Sharpe: {s_mv:.2f}<extra></extra>',
))

fig3d.update_layout(
    title='Efficient Frontier 3D',
    scene=dict(
        xaxis=dict(title='Volatility (%)', ticksuffix='%'),
        yaxis=dict(title='Return (%)',     ticksuffix='%'),
        zaxis=dict(title='Sharpe Ratio'),
    ),
    height=700,
)
fig3d.show()